# 04 — Baseline Anomaly Detection

## 1. Objective

Establish simple, interpretable anomaly-detection baselines for the SWaT dataset before applying machine-learning and deep-learning methods.

The baselines provide reference performance against which the more complex models developed in later notebooks can be evaluated.

## 2. Baseline Methods

This notebook evaluates three baseline approaches:

1. **Rule-Based Detection**
   - Detect observations that violate predefined sensor-value conditions.
   - Provides an interpretable operational baseline.

2. **Rolling Statistical Detection**
   - Detect observations whose sensor values deviate substantially from their recent temporal behaviour.
   - Uses rolling statistics while preserving chronological order.

3. **Simple Statistical Threshold**
   - Detect observations using thresholds derived from normal training data.
   - Provides a simple distribution-based reference without using attack labels during threshold construction.

## 3. Data Handling Principles

- Raw files in `data/raw` are not modified.
- Project-relative path discovery is used.
- Temporal ordering is preserved.
- The normal and attack datasets are not artificially concatenated to create a continuous time series.
- The missing-data block identified in Notebook 02 is not filled using blanket forward/backward imputation.
- Thresholds and statistical parameters are derived only from data available during model development.
- Attack labels are reserved for evaluation and are not used to construct detection thresholds.
- The final evaluation period must remain independent of threshold selection.

## 4. Relationship to Previous Notebooks

Notebook 02 established the data-quality and temporal-integrity constraints that must be respected during modelling.

Notebook 03 established exploratory characteristics of the sensor variables, including:

- substantial missingness in seven normal sensor variables;
- constant behaviour in several sensors;
- differences between normal and attack sensor distributions;
- structured relationships among sensor variables; and
- temporal and multivariate characteristics relevant to anomaly detection.

These findings guide the baseline design but do not determine anomaly thresholds by themselves.

## 5. Evaluation Principle

The baselines will be evaluated using attack labels only after their detection rules or thresholds have been established.

Performance will be interpreted as a reference point rather than as evidence that a baseline represents the underlying anomaly mechanism.

## 6. Expected Outcome

The notebook will establish reproducible baseline detection behaviour and performance that can be compared with the LSTM autoencoder, traditional machine-learning methods, and deep-learning methods developed in later notebooks.

## 7. Baseline Calibration and Evaluation Data

Baseline detection rules and statistical thresholds must be established without using attack labels or future evaluation observations.

The normal data contain a substantial missing-data block and a repeated temporal segment identified during data-quality analysis. Therefore, baseline calibration will use a clearly defined normal-data period that is suitable for parameter estimation and does not require blanket imputation.

The calibration period will be determined from the temporal structure established in Notebook 02. Evaluation data will remain temporally separate from the observations used to construct the baseline rules and thresholds.

No threshold will be selected by maximizing performance on attack labels. Attack labels will be used only after the detection procedure has been fixed, for performance evaluation.

This separation is required to prevent information leakage and to ensure that the baseline results represent a valid out-of-sample reference.

## 8. Mathematical Foundation

Let `x_t,j` denote the value of sensor `j` at time `t`.

For a baseline detector, an anomaly score `s_t` is calculated for each observation. A threshold `τ` is then used to classify the observation:

**Anomaly rule:**

`s_t > τ  →  anomaly`

`s_t ≤ τ  →  normal`

where:

- `s_t` is the anomaly score at time `t`;
- `τ` is the detection threshold.

The three baseline methods differ primarily in how the anomaly score and threshold are defined.

### 8.1 Rule-Based Detection

A rule-based detector applies an explicit condition to a sensor value.

For a sensor with acceptable limits `L_j` and `U_j`:

**Anomaly condition:**

`x_t,j < L_j` or `x_t,j > U_j`

**Normal condition:**

`L_j ≤ x_t,j ≤ U_j`

This method is highly interpretable because each detection decision can be directly traced to a predefined sensor condition.

### 8.2 Rolling Statistical Detection

For a rolling window containing the previous `w` observations, the rolling mean is:

**Rolling mean:**

`μ_t,w = (1 / w) × Σ(x_i)`

where the summation is over the previous `w` observations before time `t`.

The rolling standard deviation is:

**Rolling standard deviation:**

`σ_t,w = sqrt[Σ(x_i - μ_t,w)² / (w - 1)]`

The standardized deviation of the current observation is:

**Rolling z-score:**

`z_t = |x_t - μ_t,w| / σ_t,w`

An observation is classified as anomalous when:

`z_t > τ`

Only previous observations are used to calculate the rolling mean and standard deviation. This prevents the current observation from influencing its own reference distribution and preserves the causal temporal structure of the detector.

### 8.3 Simple Statistical Threshold

For a sensor with `n` normal calibration observations, the sample mean is:

**Sample mean:**

`x̄ = (1 / n) × Σ(x_i)`

The sample standard deviation is:

**Sample standard deviation:**

`s = sqrt[Σ(x_i - x̄)² / (n - 1)]`

The standardized deviation of an observation is:

**Statistical z-score:**

`z_t = |x_t - x̄| / s`

The observation is flagged as anomalous when:

`z_t > τ`

The mean and standard deviation must be estimated from the designated calibration data only.

### 8.4 Interpretation

These baselines make different assumptions:

- Rule-based detection relies on predefined acceptable operating conditions.
- Rolling statistical detection models deviation from recent temporal behaviour.
- Simple statistical thresholding models deviation from the overall normal calibration distribution.

The baselines are intentionally simple. Their purpose is to establish interpretable reference performance before introducing models capable of learning more complex temporal and multivariate patterns.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw" / "normal.csv").is_file()
        and (directory / "data" / "raw" / "attack.csv").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/normal.csv' and 'data/raw/attack.csv'."
    )

DATA_ROOT = PROJECT_ROOT / "data" / "raw"

normal_path = DATA_ROOT / "normal.csv"
attack_path = DATA_ROOT / "attack.csv"

# print(f"Project root: {PROJECT_ROOT}")
print(f"Normal dataset found: {normal_path.is_file()}")
print(f"Attack dataset found: {attack_path.is_file()}")

Normal dataset found: True
Attack dataset found: True


In [3]:
normal = pd.read_csv(normal_path, low_memory=False)
attack = pd.read_csv(attack_path, low_memory=False)

normal.columns = normal.columns.str.strip()
attack.columns = attack.columns.str.strip()

TIMESTAMP_COL = "Timestamp"
LABEL_COL = "Normal/Attack"

for name, dataset in [("Normal", normal), ("Attack", attack)]:
    if TIMESTAMP_COL not in dataset.columns:
        raise KeyError(f"{name} dataset is missing '{TIMESTAMP_COL}'.")

    if LABEL_COL not in dataset.columns:
        raise KeyError(f"{name} dataset is missing '{LABEL_COL}'.")

    dataset[TIMESTAMP_COL] = pd.to_datetime(
        dataset[TIMESTAMP_COL],
        errors="coerce"
    )

normal_features = set(normal.columns) - {TIMESTAMP_COL, LABEL_COL}
attack_features = set(attack.columns) - {TIMESTAMP_COL, LABEL_COL}

if normal_features != attack_features:
    raise ValueError("Normal and attack datasets have different sensor columns.")

feature_cols = [
    column
    for column in normal.columns
    if column not in (TIMESTAMP_COL, LABEL_COL)
]

print(f"Normal dataset shape: {normal.shape}")
print(f"Attack dataset shape: {attack.shape}")
print(f"Number of sensor features: {len(feature_cols)}")
print(f"Invalid normal timestamps: {normal[TIMESTAMP_COL].isna().sum()}")
print(f"Invalid attack timestamps: {attack[TIMESTAMP_COL].isna().sum()}")
print(f"Normal labels: {normal[LABEL_COL].value_counts(dropna=False).to_dict()}")
print(f"Attack labels: {attack[LABEL_COL].value_counts(dropna=False).to_dict()}")
print(
    "Normal missing sensor cells:",
    int(normal[feature_cols].isna().sum().sum())
)
print(
    "Attack missing sensor cells:",
    int(attack[feature_cols].isna().sum().sum())
)

C:\Users\ADEGOKE\AppData\Local\Temp\ipykernel_23496\789157847.py:17: UserWarning: Parsing dates in  %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dataset[TIMESTAMP_COL] = pd.to_datetime(
C:\Users\ADEGOKE\AppData\Local\Temp\ipykernel_23496\789157847.py:17: UserWarning: Parsing dates in  %d/%m/%Y %I:%M:%S %p format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dataset[TIMESTAMP_COL] = pd.to_datetime(


Normal dataset shape: (1387098, 53)
Attack dataset shape: (54621, 53)
Number of sensor features: 51
Invalid normal timestamps: 5
Invalid attack timestamps: 0
Normal labels: {'Normal': 1387098}
Attack labels: {'Attack': 54621}
Normal missing sensor cells: 6942600
Attack missing sensor cells: 0
